<a href="https://colab.research.google.com/github/lhidalgo42/3dgs/blob/main/dko_3dgs_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# dko-3dgs — pipeline completo en Colab

Reconstrucción 3D Gaussian Splatting a partir de un video walkthrough:

```
video (Google Drive) → frames → selección de nítidos → SfM (hloc GPU) → entrenamiento 3DGS → resultado a Drive
```

**Requisito:** runtime con GPU (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`).

Repo: https://github.com/lhidalgo42/3dgs

In [ ]:
!nvidia-smi

## 1. Descargar el video desde Google Drive

**Opción A (por defecto):** descarga directa de la carpeta compartida con `gdown` (requiere que la carpeta esté compartida como "cualquiera con el enlace").

In [2]:
FOLDER_URL = "https://drive.google.com/drive/folders/1wnyrTD_S8k0N2pDbll_23LTxZJvVqhZP"

!pip install -q gdown
!mkdir -p /content/video_src
!gdown --folder "$FOLDER_URL" -O /content/video_src

Retrieving folder contents
Processing file 1zu2P6Vu6UQEFMCn-lu0XH90RXfJ_jrN7 dko.MOV
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1zu2P6Vu6UQEFMCn-lu0XH90RXfJ_jrN7
From (redirected): https://drive.google.com/uc?id=1zu2P6Vu6UQEFMCn-lu0XH90RXfJ_jrN7&confirm=t&uuid=c3c2a685-42ec-47a7-8cb6-2f4ceb9f2675
To: /content/video_src/dko.MOV
100% 2.98G/2.98G [00:43<00:00, 67.8MB/s]
Download completed


**Opción B (si la carpeta es privada):** descomenta y ejecuta esta celda para montar tu Drive y copiar el video.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Ajusta la ruta a donde esté la carpeta en tu Drive:
# !mkdir -p /content/video_src
# !cp /content/drive/MyDrive/RUTA/A/LA/CARPETA/*.* /content/video_src/

In [3]:
from pathlib import Path

vids = [p for p in Path("/content/video_src").rglob("*")
        if p.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv")]
assert vids, "No se encontró ningún video en /content/video_src"
VIDEO = str(vids[0])
print("Video:", VIDEO)

Video: /content/video_src/dko.MOV


## 2. Extraer frames y seleccionar los más nítidos

Se extraen todos los frames y se conserva el más nítido (varianza del Laplaciano) de cada ventana de `WIN` frames — igual que `select_sharp.py` del repo. Apunta a ~200–300 imágenes finales; ajusta `WIN` según la duración del video.

In [ ]:
!mkdir -p /content/candidates
!ffmpeg -hide_banner -loglevel error -i "$VIDEO" -qscale:v 2 /content/candidates/%05d.jpg

n_frames = len(list(Path("/content/candidates").glob("*.jpg")))
print(n_frames, "frames extraídos")

In [ ]:
import cv2, shutil

WIN = max(1, n_frames // 240)  # apunta a ~240 imágenes; fíjalo a mano si prefieres
src, dst = Path("/content/candidates"), Path("/content/data/input")
dst.mkdir(parents=True, exist_ok=True)

frames = sorted(src.glob("*.jpg"))
kept = 0
for i in range(0, len(frames), WIN):
    window = frames[i:i + WIN]
    best = max(window, key=lambda f: cv2.Laplacian(
        cv2.imread(str(f), cv2.IMREAD_GRAYSCALE), cv2.CV_64F).var())
    shutil.copy(best, dst / f"{kept:04d}.jpg")
    kept += 1
print(f"ventana={WIN}: conservadas {kept} de {len(frames)}")

## 3. Structure from Motion con hloc (ALIKED + LightGlue, GPU)

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization /content/Hierarchical-Localization
!pip install -q -e /content/Hierarchical-Localization
!pip install -q pycolmap

In [ ]:
import pycolmap
from hloc import extract_features, match_features, reconstruction

images = Path("/content/data/input")
out = Path("/content/hloc_out")
out.mkdir(exist_ok=True)

names = sorted(p.name for p in images.glob("*.jpg"))
OVERLAP = 15
pairs = out / "pairs.txt"
with open(pairs, "w") as f:
    for i, a in enumerate(names):
        for j in range(i + 1, min(i + 1 + OVERLAP, len(names))):
            f.write(f"{a} {names[j]}\n")
print(f"{len(names)} imágenes, pares secuenciales escritos")

feat_conf = extract_features.confs["aliked-n16"]
match_conf = match_features.confs["aliked+lightglue"]
features = extract_features.main(feat_conf, images, out)
matches = match_features.main(match_conf, pairs, feat_conf["output"], out)

model = reconstruction.main(
    out / "sfm", images, pairs, features, matches,
    camera_mode=pycolmap.CameraMode.SINGLE,
    image_options=dict(camera_model="OPENCV"),
)
print("Imágenes registradas:", model.num_reg_images() if model else 0)

## 4. Undistorsionar y armar el layout que espera 3DGS

3DGS necesita `data/images/` (imágenes sin distorsión) + `data/sparse/0/` (modelo COLMAP).

In [ ]:
import shutil

data = Path("/content/data")
pycolmap.undistort_images(
    output_path=str(data),
    input_path=str(out / "sfm"),
    image_path=str(images),
    output_type="COLMAP",
)

sparse0 = data / "sparse" / "0"
sparse0.mkdir(parents=True, exist_ok=True)
for f in (data / "sparse").iterdir():
    if f.is_file():
        shutil.move(str(f), sparse0 / f.name)
print(sorted(p.name for p in sparse0.iterdir()))

## 5. Instalar gaussian-splatting y entrenar

La compilación de los submódulos CUDA tarda unos minutos. El entrenamiento a 7000 iteraciones toma ~20–40 min en una T4.

In [ ]:
!git clone --quiet --recursive https://github.com/graphdeco-inria/gaussian-splatting /content/gaussian-splatting
!pip install -q plyfile \
    /content/gaussian-splatting/submodules/diff-gaussian-rasterization \
    /content/gaussian-splatting/submodules/simple-knn \
    /content/gaussian-splatting/submodules/fused-ssim

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s /content/data -m /content/output/dko3d \
    --iterations 7000 --save_iterations 7000 --test_iterations -1

## 6. Guardar el resultado en Google Drive

Copia el modelo entrenado (`point_cloud.ply` + cámaras) a tu Drive.

In [ ]:
from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/dko3d_output
!cp -r /content/output/dko3d /content/drive/MyDrive/dko3d_output/
print("Resultado copiado a MyDrive/dko3d_output/dko3d")

## 7. Visualizar

Descarga `dko3d/point_cloud/iteration_7000/point_cloud.ply` y ábrelo en [SuperSplat](https://playcanvas.com/supersplat/editor) (arrastra el archivo al navegador) o cualquier visor de gaussian splats.